# Agentic RAG

## What is Agentic RAG?

Agentic RAG is a system whereby a variety of retrieval agents are available to retrieve the data needed to answer the user question. The basic components of an Agentic RAG system are:
- *Retriever router*: A function that takes in the user question(s) and returns the best retriever(s) to use
- *Retriever agents*: The actual retrievers that can be used to retrieve the data needed to answer the user question(s)
- *Answer critic*: A function that takes in the answers from the retrievers and checks if the original question is answered correctly

### Retriever Agents

Can be
- Vector search over a knowledge graph embedding
- Keyword search over a knowledge graph
... etc

### Retriever Router

How the router makes this decision can vary, but usually an LLM is used to make this decision.

### Answer Critic

The answer critic is a blocking function that can stop the answer from being returned to the user if the answer is not correct or is incomplete.

If an incomplete or incorrect answer is blocked, the answer critic should generate a new question that can be used to retrieve the correct answer and go through another round of retrieving the correct answer. It might be that the correct answer is not available in the data source, so there needs to be some exit criteria from this loop; the answer critic should be able to handle that and return a message to the user that the answer is not available in such cases.


## Why Do We Need Agentic RAG?

One area where agentic RAG is useful is when we have a variety of data sources and we want to use the best data source for the job. Another common usage is when the
data source is very broad or complex and we need specialized retrievers to retrieve the data we need consistently.

## How to Implement Agentic RAG?

In [1]:
import sys
import os
sys.path.append(os.path.abspath(".."))

from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from importlib import reload
import json

from utils.utils import chat, tool_choice

### Implementing Retriever Tools

Before we can route the user input to be handled by the right retriever(s), we need to have the retrievers available for the router to choose from.

In this example, we will use a list of retrievers:
- 2 retrievers that use Cypher templates to get movies by title and movies by actor name and
- 1 retriever that uses text2cypher for all other questions.

We will load these from `ch05_tools.py` and then we will implement the router and answer critic here.
- `movie_info_by_title`: A retriever that uses a Cypher template to get movie information by title
- `movie_info_by_actor`: A retriever that uses a Cypher template to get movie information by actor name
- `text2cypher`: A retriever that uses text2cypher to convert the user question into a Cypher query and then executes the query to get the answer


We need to be careful with how we describe the retriever to the LLM. We need to make sure the LLM understands the retriever and can make a decision on which retriever to use.

Note that the LLM can’t make actual calls to your retrievers; it can only make a decision on which retriever to use and what parameters to pass to the retriever.

We also want to keep a generic retriever tool in our agentic RAG system if the answer to the question is already given within the question or other parts of the context,
- `answer_given`: A retriever that simply returns the answer given in the question or other parts of the context without calling any external data source. 

### Implementing the Retriever Router

The goal of the retriever router is to take in the user question and return the best retriever(s) to use for that question.

When the LLM returns the best retriever(s) to use, the system needs to make the call to the retriever(s).

In [3]:
def handle_tool_calls(tools: dict[str, any], llm_tool_calls: list[dict[str, any]]):
    output = []

    if llm_tool_calls:
        for tool_call in llm_tool_calls:
            function_to_call = tools[tool_call.function.name]['function']
            function_args = json.loads(tool_call.function.arguments)
            res = function_to_call(**function_args)
            output.append(res)

    return output

The `llm_tool_calls` is a list of the tools the LLM has decided to use and the arguments to pass to the tool. The shape of the `llm_tool_calls` argument looks like:
```json
[
    {
        "function": {
            "name": "answer_given",
            "arguments": "{\"answer\": \"Dave Smith\"}"
        }
    }
]
```

We will send the questions to the LLM one by one in sequence so that we can use the answers from the previous questions to rewrite the next question if needed. This can be useful if the user asks a follow-up question that is dependent on the answer to the previous question.

In [4]:
query_update_prompt = """
    You are an expert at updating questions to make the them ask for one thing only, more atomic, specific and easier to find the answer for.
    You do this by filling in missing information in the question, with the extra information provided to you in previous answers. 
    
    You respond with the updated question that has all information in it.
    Only edit the question if needed. If the original question already is atomic, specific and easy to answer, you keep the original.
    Do not ask for more information than the original question. Only rephrase the question to make it more complete.
    
    JSON template to use:
    {
        "question": "question1"
    }
"""

The query updater is called with the original question and the answers from the retrievers. The output is the updated question, and we instruct the LLM to return the updated question in a JSON format.

In [5]:
def query_update(input: str, answers: list[any]) -> str:
    messages = [
        {"role": "system", "content": query_update_prompt},
        *answers,
        {"role": "user", "content": f"The user question to rewrite: '{input}'"}
    ]

    config = {"response_format": {"type": "json_object"}}

    output = chat(messages=messages, model="gpt-4.1", config=config)

    try:
        return json.loads(output)['question']
    except json.JSONDecodeError:
        print("Error decoding JSON")

    return []

The final piece in the retriever router is to call the LLM with the questions and the available tools.

First we need to prepare the tool list:

In [6]:
import ch05_tools

tools = {
    "movie_info_by_title": {
        "description": ch05_tools.movie_info_by_title_description,
        "function": ch05_tools.movie_info_by_title
    },
    "movies_info_by_actor": {
        "description": ch05_tools.movies_info_by_actor_description,
        "function": ch05_tools.movies_info_by_actor
    },
    "text2cypher": {
        "description": ch05_tools.text2cypher_description,
        "function": ch05_tools.text2cypher
    },
    "answer_given": {
        "description": ch05_tools.answer_given_description,
        "function": ch05_tools.answer_given
    }
}


tool_picker_prompt = """
    Your job is to chose the right tool needed to respond to the user question. 
    The available tools are provided to you in the prompt.
    Make sure to pass the right and the complete arguments to the chosen tool.
"""

Next we need to build the function to call LLM:

In [7]:
def route_question(question: str, tools: dict[str, any], answers: list[dict[str, str]]):
    llm_tool_calls = tool_choice(
        [
            {
                "role": "system",
                "content": tool_picker_prompt,
            },
            *answers,
            {
                "role": "user",
                "content": f"The user question to find a tool to answer: '{question}'",
            },
        ],
        model = "gpt-4.1",
        tools=[tool["description"] for tool in tools.values()],
    )
    return handle_tool_calls(tools, llm_tool_calls)

The `handle_tool_calls` function will make the actual calls to the retriever tools based on the `llm_tool_calls` list returned by the LLM.

The final piece of the retriever router is to tie all previous parts together and go all the way from the user input to the answer. This needs to be done in a loop that goes through all questions and that we update the questions with the new information as we go along.

In [8]:
def handle_user_input(input: str, answers: list[dict[str, str]] = []):
    updated_question = query_update(input, answers)

    response = route_question(updated_question, tools, answers)
    answers.append(
        {"role": "assistant", "content": f"For the question: '{updated_question}', we have the answer: '{json.dumps(response)}'"}
    )
    return answers

With this in place, we have a complete agentic RAG system that can take in user input and return the answer to the user. The system is built in a way that it can be extended with more retrievers as needed.

### Implementing the Answer Critic

The goal of the answer critic is to take all answers from the retrievers and check if the original question is answered correctly.

In [9]:
answer_critique_prompt = """
    You are an expert at identifying if questions has been fully answered or if there is an opportunity to enrich the answer.
    The user will provide a question, and you will scan through the provided information to see if the question is answered.
    If anything is missing from the answer, you will provide a set of new questions that can be asked to gather the missing information.
    All new questions must be complete, atomic and specific.
    However, if the provided information is enough to answer the original question, you will respond with an empty list.

    JSON template to use for finding missing information:
    {
        "questions": ["question1", "question2"]
    }
"""

In [10]:
def critique_answers(question: str, answers: list[dict[str, str]]) -> list[str]:
    messages = [
        {
            "role": "system",
            "content": answer_critique_prompt
        },
        *answers,
        {
            "role": "user",
            "content": f"The original user question to answer: '{question}'",
        },
    ]

    config = {"response_format": {"type": "json_object"}}
    output = chat(messages=messages, model="gpt-4.1", config=config)

    try:
        return json.loads(output)['questions']
    except json.JSONDecodeError:
        print("Error decoding JSON")

    return []

### Tying It All Together

In [11]:
main_prompt = """
    Your job is to help the user with their questions.
    You will receive user questions and information needed to answer the questions
    If the information is missing to answer part of or the whole question, you will say that the information 
    is missing. You will only use the information provided to you in the prompt to answer the questions.
    You are not allowed to make anything up or use external information.
"""

In [13]:
def main(input: str):
    answers = handle_user_input(input)

    critique = critique_answers(input, answers)

    if critique:
        answers = handle_user_input(" ".join(critique), answers)

    llm_response = chat(
        [
            {"role": "system", "content": main_prompt},
            *answers,
            {"role": "user", "content": f"The user question to answer: '{input}'"},
        ],
        model="gpt-4.1"
    )

    return llm_response

The main function runs the user input through the agentic RAG system and returns the answer to the user. If the answer is not complete or is incorrect, the critique function will return a list of new questions that can be asked to gather the missing information.

We only critique the answers once; if the answers are still incomplete or incorrect after the critique, we return the answers to the user as is and rely on the LLM to let the user know what’s incomplete.

In [ ]:
response = main("Who's the main actor in the movie Matrix and what other movies is that person in?")
print(f"Main response: {response}")

In [ ]:
next_res = main("How many directors and producers are in the database?")
print(f"Next response: {next_res}")